# AE label of one shot

Type a shot number and run all. Shows the 1 ms AE label from the all-shot database
(`/projects/.../ae_seldnet_fdp_1ms`), the model probability behind it, and the CO2
spectrogram when a local copy exists (tier shots only; other inputs are deleted after labeling).

In [ ]:
SHOT = 185786

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DB = Path("/projects/EKOLEMEN/tokamaksi/diiid/release/alfven_eigenmode/ae_seldnet_fdp_1ms")
CACHE = Path("/scratch/gpfs/EKOLEMEN/nc1514/FusionAIHub/.cache/labeler")
WORK = CACHE / "ae_fdp_1ms/work"    # model probabilities, optional
SPECS = CACHE / "ae_tiers/specs"    # spectrograms, tier shots only, optional
if not (DB / "grid_1ms/index.csv").exists():  # until the all-shot run is published
    DB, WORK = CACHE / "ae_fdp_preview/db", CACHE / "ae_tiers/work"
    print(f"database not published yet; using the local preview (stable model, 498 shots): {DB}")
shots = pd.read_parquet(DB / "shots.parquet").set_index("shot")
grid = pd.read_csv(DB / "grid_1ms/index.csv").set_index("shot")


def show(shot, xlim=None):
    if shot not in grid.index:
        print(f"{shot}: no label ({shots.status.get(shot, 'not a plasma shot in the d3drdb snapshot')})")
        return
    g = grid.loc[shot]
    n = g.end_ms - g.start_ms
    ae = np.unpackbits(np.fromfile(DB / f"grid_1ms/{shot // 1000}/{shot}.bin", np.uint8), count=n)
    t = g.start_ms + np.arange(n)
    spec, work = SPECS / f"{shot}.npy", WORK / f"{shot}.npz"
    panels = [p for p, ok in (("spec", spec.exists()), ("prob", work.exists())) if ok] + ["ae"]
    fig, axes = plt.subplots(len(panels), 1, sharex=True, squeeze=False, figsize=(11, 0.8 + 1.5 * len(panels)),
                             height_ratios=[3 if p == "spec" else 1 for p in panels])
    ax = dict(zip(panels, axes[:, 0]))
    if "spec" in ax:
        s = np.load(spec, mmap_mode="r").mean(axis=1, dtype=np.float32).T  # chords averaged, (freq, time)
        ts = np.load(spec.with_suffix(".meta.npz"))["t_ms"]
        ax["spec"].imshow(s, origin="lower", aspect="auto", cmap="magma", extent=(ts[0], ts[-1], 80, 250),
                          vmin=np.percentile(s, 5), vmax=np.percentile(s, 99.5))
        ax["spec"].set_ylabel("CO2 kHz")
    if "prob" in ax:
        w = np.load(work)
        ax["prob"].plot(w["t_first_ms"] + w["hop_ms"] * np.arange(w["n_columns"]), w["prob"], lw=0.5, c="k")
        ax["prob"].axhline(0.5, c="r", lw=0.5, ls="--")
        ax["prob"].set(ylim=(0, 1), ylabel="model p")
    ax["ae"].fill_between(t, ae, step="post", color="seagreen", lw=0)
    ax["ae"].set(ylim=(0, 1), yticks=[0, 1], ylabel="AE", xlabel="time (ms)", xlim=xlim or (g.start_ms, g.end_ms))
    axes[0, 0].set_title(f"{shot}: AE in {ae.sum():,} of {n:,} ms (labels on [{g.start_ms}, {g.end_ms}) ms)")
    plt.show()

In [ ]:
show(SHOT)  # show(SHOT, xlim=(1000, 2000)) to zoom